In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import norm, ttest_ind
from datetime import datetime
import os

##### ***Задача 1.*** Оценить необходимый размер групп
  

Допустим, мы хотим провести эксперимент, в который попадают клиенты, совершившие покупку во время эксперимента.

Метрика — средняя выручка с пользователя за время эксперимента;
Продолжительность — одна неделя;
Уровень значимости — 0.05;
Допустимая вероятность ошибки II рода — 0.1;
Ожидаемый эффект — 20 рублей.
Оцените необходимый размер групп по данным о покупках за неделю с 21 по 28 февраля. Обратим внимание, что в выборку попадают события из полуинтервала [datetime(2022, 2, 21), datetime(2022, 2, 28)).

Для решения используйте данные из файла 2022-04-01T12_df_sales.csv.

В качестве ответа введите необходимый размер групп, округлённый до целого числа десятков round(x, -1).

In [7]:
URL_BASE = 'https://raw.githubusercontent.com/ab-courses/simulator-ab-datasets/main/2022-04-01/'

def read_database(file_name):
    return pd.read_csv(os.path.join(URL_BASE, file_name))
df_sales = read_database('2022-04-01T12_df_sales.csv')
df_sales['date'] = pd.to_datetime(df_sales['date'])

In [9]:
begin_date_pred = datetime(2022, 2, 21)
end_date_pred = datetime(2022, 2, 28)
df_sales_pred = df_sales[(df_sales['date'] >= begin_date_pred) & (df_sales['date'] < end_date_pred)]
df_sales_pred.head()

,sale_id,date,count_pizza,count_drink,price,user_id
63207,1063208,2022-02-21 10:02:02,4,1,3030,fcaa0f
63208,1063209,2022-02-21 10:04:43,3,1,2250,8d8445
63209,1063210,2022-02-21 10:05:40,1,0,780,e21af3
63210,1063211,2022-02-21 10:09:01,4,0,2880,e23104
63211,1063212,2022-02-21 10:11:20,2,1,1620,748932


In [24]:
df_sales_pred.shape

(25347, 6)

In [12]:
arrpu = df_sales_pred.groupby('user_id')['price'].sum()

In [14]:
arrpu_mean = arrpu.mean()
arrpu_mean

np.float64(1234.6871980676328)

In [20]:
arrpu_mean_2 = df_sales_pred['price'].sum()/df_sales_pred.user_id.nunique()
arrpu_mean_2

np.float64(1234.6871980676328)

In [15]:
arrpu_std = arrpu.std()
arrpu_std

np.float64(811.1803387604912)

MDE = 20 рублей

a=0.05

b=0.1

In [16]:
def get_sample_size_abs(epsilon, std, alpha=0.05, beta=0.2):
    t_alpha = norm.ppf(1 - alpha / 2, loc=0, scale=1)
    t_beta = norm.ppf(1 - beta, loc=0, scale=1)
    z_scores_sum_squared = (t_alpha + t_beta) ** 2
    sample_size = int(
        np.ceil(
            z_scores_sum_squared * (2 * std ** 2) / (epsilon ** 2)
        )
    )
    return sample_size

In [22]:
alpha = 0.05
beta = 0.1
mde = 20 
std = arrpu_std
mean = arrpu_mean

sample_size = get_sample_size_abs(mde, std, alpha=alpha, beta=beta)
print(f'sample_size = {sample_size}')

sample_size = 34571


In [23]:
print(f'sample_size = {round(sample_size, -1)}')

sample_size = 34570


##### ***Задача 2. MDE***
  

В прошлом задании получилось, что необходимый размер групп больше имеющихся данных за одну неделю. Какой минимальный эффект мы можем отловить с теми же вероятностями ошибок на данных
с 21 по 28 февраля?

Ответ округлите до целого значения.



In [39]:
df_sales_pred.shape

#  есть только 25347  записей - меньше чем нужно для эксперимента, чтобы задетектить mde=20 (sample_size = 34570)

(25347, 6)

In [32]:
size_1_group = int(df_sales_pred.shape[0] / 2)
size_1_group

12673

In [40]:
def get_minimal_determinable_effect(std, sample_size, alpha=0.05, beta=0.2):
    t_alpha = norm.ppf(1 - alpha / 2, loc=0, scale=1)
    t_beta = norm.ppf(1 - beta, loc=0, scale=1)
    disp_sum_sqrt = (2 * (std ** 2)) ** 0.5
    mde = (t_alpha + t_beta) * disp_sum_sqrt / np.sqrt(sample_size)
    return mde

In [37]:
mde = get_minimal_determinable_effect(std, df_sales_pred.shape[0] // 2, alpha=alpha, beta=beta)
mde

np.float64(33.032451460660994)

##### ***Задача 3.*** Функция оценки размера выборки

In [47]:
import numpy as np
import pandas as pd
import scipy.stats as stats

def estimate_sample_size(metrics, effect, alpha, beta):
    mean = metrics['metric'].mean()
    std = np.std(metrics['metric'])
    
    mde = mean * effect / 100
    
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(1 - beta)
    
    sample_size = 2 * ((z_alpha + z_beta) * std / mde) ** 2
    
    if metrics['user_id'].duplicated().any():
        values_per_user = len(metrics) / metrics['user_id'].nunique()
        sample_size = sample_size / values_per_user
    
    return int(np.ceil(sample_size))

In [48]:
metrics = pd.DataFrame({
    'user_id': np.arange(100),
    'metric': np.linspace(500, 1490, 100)
})
effect, alpha, beta = 3, 0.05, 0.1
sample_size = estimate_sample_size(metrics, effect, alpha, beta)
print(sample_size)
# sample_size = 1966

metrics = pd.DataFrame({
    'user_id': np.arange(100) % 30,
    'metric': np.linspace(500, 1490, 100)
})
effect, alpha, beta = 3, 0.05, 0.1
sample_size = estimate_sample_size(metrics, effect, alpha, beta)
print(sample_size)
# sample_size = 590

1966
590
